# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides guidance for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets in the dataset
print("Available Record Sets (by @id):")
record_sets = dataset.metadata.record_sets
for i, rs in enumerate(record_sets):
    print(f"  {i+1}. {rs['@id']} - {rs.get('name', '(no name)')}")

# Inspecting the first record set
main_record_set_id = record_sets[0]['@id'] if len(record_sets) > 0 else None
print(f"\nFields for record set: {main_record_set_id}")
if main_record_set_id:
    fields = dataset.metadata.get_record_set_by_id(main_record_set_id)['fields']
    for f in fields:
        print(f"  - @id: {f['@id']} | name: {f.get('name', '')} | dataType: {f.get('dataType', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets that can be read
dataframes = {}
for record_set in record_sets:
    rs_id = record_set['@id']
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set {rs_id}.")
    except Exception as e:
        print(f"[Warning] Could not load records for record set {rs_id}: {e}")

# Select the main record set for further analysis
chosen_record_set_id = main_record_set_id
if chosen_record_set_id and chosen_record_set_id in dataframes:
    print(f"Columns in {chosen_record_set_id}:")
    print(dataframes[chosen_record_set_id].columns.tolist())
    display(dataframes[chosen_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, and grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Identify a numeric field for analysis -- use its @id, e.g., cr:MW for Molecular Weight
numeric_field = None
group_field = None
field_info = dataset.metadata.get_record_set_by_id(chosen_record_set_id)['fields']
for f in field_info:
    # Try to get a likely numeric field, like Mass or Abundance
    if f.get('dataType', '').lower() in ['number', 'float', 'integer']:
        if ('abundance' in f.get('name', '').lower() or
            'mw' in f.get('name', '').lower() or
            'mass' in f.get('name', '').lower() or
            'peptide' in f.get('name', '').lower()):
            numeric_field = f['@id']
            break
# Fallback if not found
if numeric_field is None and field_info:
    for f in field_info:
        if f.get('dataType', '').lower() in ['number', 'float', 'integer']:
            numeric_field = f['@id']
            break
    
# Select a grouping field, e.g. Protein Description, if present
for f in field_info:
    if 'description' in f.get('name', '').lower():
        group_field = f['@id']
        break

if numeric_field:
    print(f"Using numeric field for analysis: {numeric_field}")
    # If '@id' is not a column name, try fallback to display name
    df = dataframes[chosen_record_set_id]
    col_candidates = [numeric_field, numeric_field.split(':')[-1]]
    colname = None
    for c in col_candidates:
        if c in df.columns:
            colname = c
            break
    if not colname:
        colname = df.columns[0] # fallback

    # Try to coerce to numeric just in case
    df[colname] = pd.to_numeric(df[colname], errors='coerce')
    # Filtering
    threshold = np.nanmedian(df[colname])
    filtered_df = df[df[colname] > threshold].copy()
    print(f"Filtered records with {colname} > {threshold} (median): {len(filtered_df)} records")
    # Normalization
    filtered_df[f"{colname}_normalized"] = (
        (filtered_df[colname] - filtered_df[colname].mean()) / filtered_df[colname].std()
    )
    print(filtered_df[[colname, f"{colname}_normalized"]].head())
    # Grouping
    if group_field:
        g_candidates = [group_field, group_field.split(':')[-1]]
        g_colname = None
        for g in g_candidates:
            if g in df.columns:
                g_colname = g
                break
        if g_colname:
            grouped_df = filtered_df.groupby(g_colname)[colname].mean().reset_index()
            print(f"Grouped mean of {colname} by {g_colname}:")
            print(grouped_df.head())
        else:
            print("No valid grouping column found.")
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field
if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[colname].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {colname}")
    plt.xlabel(colname)
    plt.ylabel("Count")
    plt.show()

    # If grouped_df exists, plot top groups if applicable
    if 'grouped_df' in locals() and grouped_df.shape[1] >= 2:
        plt.figure(figsize=(10,5))
        top_n = min(10, len(grouped_df))
        sns.barplot(
            data=grouped_df.sort_values(colname, ascending=False)[:top_n],
            x=grouped_df.columns[0], y=colname
        )
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Top {top_n} groups by mean {colname}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.
